In [88]:
import pandas as pd
from sklearn.cluster import KMeans

In [52]:
mov = pd.read_csv("/Users/fangsiyu/Desktop/Kolding Hackathon 2026/Movement.csv")
sd = pd.read_csv("/Users/fangsiyu/Desktop/Kolding Hackathon 2026/StayDuration.csv")

mov
sd

,sensor,category,sum_ms,min_ms,max_ms,objects,timestamp
0,12-Zone 1 - Duration of occurrence - Table,wheelchair,21360,6240,15120,2,2026-02-09T01:00:00+01:00
1,12-Zone 1 - Duration of occurrence - Table,van,1369760,3240,329360,48,2026-02-09T01:00:00+01:00
2,12-Zone 1 - Duration of occurrence - Table,scooter,19240,3200,9240,3,2026-02-09T01:00:00+01:00
3,12-Zone 1 - Duration of occurrence - Table,pram,286960,3040,20640,47,2026-02-09T01:00:00+01:00
4,12-Zone 1 - Duration of occurrence - Table,pedestrian,38287162,3040,597560,3097,2026-02-09T01:00:00+01:00
...,...,...,...,...,...,...,...
2186,12-Zone 1 - Duration of occurrence - Table,light,21360,4080,6600,4,2026-05-08T02:00:00+02:00
2187,12-Zone 1 - Duration of occurrence - Table,car trailer,3160,3160,3160,1,2026-05-08T02:00:00+02:00
2188,12-Zone 1 - Duration of occurrence - Table,car,4093915,3040,8520,809,2026-05-08T02:00:00+02:00
2189,12-Zone 1 - Duration of occurrence - Table,bicycle,35240,3040,8000,7,2026-05-08T02:00:00+02:00


In [50]:
grouped_by_sensor = mov.groupby(['timestamp', 'sensor', 'category', 'direction'])['amount'].sum().unstack(fill_value=0)

grouped_by_sensor['net_inflow'] = grouped_by_sensor['IN'] - grouped_by_sensor['OUT']

grouped_by_sensor = grouped_by_sensor.reset_index()
grouped_by_sensor.columns.name = None
grouped_by_sensor.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/mov_net_flow_by_gate.csv', index=False, encoding='utf-8-sig')
grouped_by_sensor

,timestamp,sensor,category,IN,OUT,net_inflow
0,2026-02-09T01:00:00+01:00,1.1,animal,3,8,-5
1,2026-02-09T01:00:00+01:00,1.1,bicycle,18,27,-9
2,2026-02-09T01:00:00+01:00,1.1,car,4,7,-3
3,2026-02-09T01:00:00+01:00,1.1,light,0,1,-1
4,2026-02-09T01:00:00+01:00,1.1,motorcycle,1,0,1
...,...,...,...,...,...,...
2904,2026-05-08T02:00:00+02:00,3.1,light,7,3,4
2905,2026-05-08T02:00:00+02:00,3.1,motorcycle,0,1,-1
2906,2026-05-08T02:00:00+02:00,3.1,pedestrian,52,22,30
2907,2026-05-08T02:00:00+02:00,3.1,pram,0,2,-2


In [81]:
mask = mov['sensor'] != "3.1"
n_mov = mov[mask]

grouped_whole_zone = n_mov.groupby(['timestamp', 'category', 'direction'])['amount'].sum().unstack(fill_value=0)

grouped_whole_zone['net_inflow'] = grouped_whole_zone['IN'] - grouped_whole_zone['OUT']

grouped_whole_zone = grouped_whole_zone.reset_index()
grouped_whole_zone.columns.name = None
# grouped_whole_zone.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/mov_net_flow_by_mix_gate_without3-1.csv', index=False, encoding='utf-8-sig')
grouped_whole_zone

,timestamp,category,IN,OUT,net_inflow
0,2026-02-09T01:00:00+01:00,animal,3,8,-5
1,2026-02-09T01:00:00+01:00,bicycle,31,36,-5
2,2026-02-09T01:00:00+01:00,car,53,68,-15
3,2026-02-09T01:00:00+01:00,light,3,1,2
4,2026-02-09T01:00:00+01:00,motorcycle,2,3,-1
...,...,...,...,...,...
1016,2026-05-08T02:00:00+02:00,motorcycle,0,1,-1
1017,2026-05-08T02:00:00+02:00,pedestrian,60,30,30
1018,2026-05-08T02:00:00+02:00,pram,0,2,-2
1019,2026-05-08T02:00:00+02:00,tractor,1,0,1


In [ ]:
mov_p = grouped_whole_zone.pivot(index='timestamp', columns='category', values='net_inflow').fillna(0)

mov_p = mov_p.reset_index()
mov_p.columns.name = None
mov_p = mov_p.add_prefix('mov_net_')
mov_p.columns.name = None

mov_p
# final_clustered_days.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/K_daily_buses_th.csv', index=False, encoding='utf-8-sig')

# print(matrix_daily[['timestamp', 'cluster_label']].head(20))

,mov_net_timestamp,mov_net_animal,mov_net_bicycle,mov_net_bus,mov_net_car,mov_net_car trailer,mov_net_heavy,mov_net_light,mov_net_motorcycle,mov_net_pedestrian,mov_net_pram,mov_net_scooter,mov_net_tractor,mov_net_tram,mov_net_truck trailer,mov_net_van,mov_net_wheelchair
0,2026-02-09T01:00:00+01:00,-5.0,-5.0,0.0,-15.0,0.0,0.0,2.0,-1.0,-57.0,2.0,-3.0,0.0,0.0,0.0,-1.0,0.0
1,2026-02-10T01:00:00+01:00,10.0,-28.0,0.0,-75.0,-1.0,0.0,14.0,0.0,-262.0,13.0,3.0,-1.0,0.0,-1.0,12.0,0.0
2,2026-02-11T01:00:00+01:00,-3.0,-27.0,1.0,-117.0,2.0,0.0,22.0,-3.0,-387.0,0.0,3.0,0.0,0.0,0.0,-35.0,1.0
3,2026-02-12T01:00:00+01:00,5.0,-15.0,2.0,-71.0,-1.0,0.0,1.0,2.0,-309.0,-5.0,-2.0,1.0,0.0,0.0,-6.0,2.0
4,2026-02-13T01:00:00+01:00,2.0,-14.0,1.0,-98.0,-3.0,0.0,15.0,-3.0,-721.0,2.0,-2.0,0.0,0.0,-1.0,-10.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,2026-05-04T02:00:00+02:00,-4.0,-1.0,0.0,-2.0,-1.0,0.0,6.0,0.0,-60.0,2.0,0.0,0.0,0.0,0.0,6.0,0.0
85,2026-05-05T02:00:00+02:00,-3.0,-3.0,0.0,-19.0,1.0,0.0,0.0,-2.0,-34.0,-12.0,2.0,0.0,0.0,0.0,9.0,0.0
86,2026-05-06T02:00:00+02:00,-4.0,-8.0,0.0,9.0,1.0,0.0,7.0,1.0,-42.0,0.0,-2.0,0.0,0.0,0.0,18.0,0.0
87,2026-05-07T02:00:00+02:00,3.0,-3.0,0.0,-24.0,1.0,0.0,6.0,0.0,57.0,-2.0,0.0,0.0,0.0,0.0,16.0,0.0


In [99]:
sd_mask = sd['sensor'] == '12-Zone 1 - Duration of occurrence - Table'
zone1_df = sd[sd_mask].copy()

pivot_df = zone1_df.pivot_table(
    index='timestamp', 
    columns='category', 
    values= ['sum_ms', 'min_ms', 'max_ms'], 
    aggfunc={
        'sum_ms': 'sum',
        'min_ms': 'min',
        'max_ms': 'max'
    }
).fillna(0)

pivot_df.columns = [f"dur_{metric}_{category}" for metric, category in pivot_df.columns]
final_zone1_all_metrics = pivot_df.reset_index()
final_zone1_all_metrics.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/dur_zone1_pivot.csv', index=False, encoding='utf-8-sig')

final_zone1_all_metrics

,timestamp,dur_max_ms_animal,dur_max_ms_bicycle,dur_max_ms_bus,dur_max_ms_car,dur_max_ms_car trailer,dur_max_ms_caravan,dur_max_ms_heavy,dur_max_ms_light,dur_max_ms_motorcycle,...,dur_sum_ms_light,dur_sum_ms_motorcycle,dur_sum_ms_pedestrian,dur_sum_ms_pram,dur_sum_ms_scooter,dur_sum_ms_tractor,dur_sum_ms_tram,dur_sum_ms_truck trailer,dur_sum_ms_van,dur_sum_ms_wheelchair
0,2026-02-09T01:00:00+01:00,120640.0,21200.0,0.0,208640.0,0.0,0.0,0.0,15720.0,15640.0,...,44280.0,226320.0,38287162.0,286960.0,19240.0,0.0,0.0,0.0,1369760.0,21360.0
1,2026-02-10T01:00:00+01:00,46000.0,41600.0,0.0,41840.0,42480.0,0.0,3480.0,49840.0,17720.0,...,3936715.0,139280.0,83449113.0,4121160.0,190040.0,62000.0,26160.0,1247999.0,2834835.0,163999.0
2,2026-02-11T01:00:00+01:00,33600.0,27480.0,6520.0,32600.0,15160.0,0.0,0.0,32320.0,33440.0,...,3011197.0,102320.0,73799138.0,922760.0,210800.0,19040.0,0.0,0.0,3503605.0,27880.0
3,2026-02-12T01:00:00+01:00,26160.0,29320.0,13480.0,27160.0,9360.0,0.0,4120.0,28040.0,21040.0,...,1277519.0,97000.0,60985504.0,1284757.0,132839.0,70480.0,0.0,0.0,1207470.0,479640.0
4,2026-02-13T01:00:00+01:00,25240.0,19600.0,22880.0,24040.0,24240.0,0.0,0.0,23240.0,16960.0,...,1088520.0,388799.0,65567301.0,664879.0,528959.0,41600.0,0.0,598278.0,4336127.0,2485239.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,2026-05-04T02:00:00+02:00,8240.0,8640.0,0.0,8640.0,3920.0,0.0,0.0,8600.0,8640.0,...,501837.0,2358548.0,13388811.0,489600.0,211240.0,0.0,0.0,0.0,2025195.0,34360.0
85,2026-05-05T02:00:00+02:00,8600.0,8240.0,0.0,8600.0,7560.0,0.0,0.0,8600.0,8600.0,...,193640.0,3764696.0,12870173.0,578999.0,139240.0,60800.0,0.0,0.0,740839.0,42640.0
86,2026-05-06T02:00:00+02:00,8320.0,8360.0,0.0,8600.0,7960.0,0.0,0.0,8600.0,8560.0,...,813397.0,3209153.0,18026354.0,814919.0,40120.0,10200.0,0.0,0.0,436720.0,50480.0
87,2026-05-07T02:00:00+02:00,8560.0,8360.0,0.0,8560.0,8240.0,0.0,0.0,8480.0,8560.0,...,659399.0,4394512.0,21003916.0,1087917.0,77520.0,0.0,0.0,0.0,159959.0,84320.0


In [100]:
temp_df = pd.merge(final_zone1_all_metrics, mov_p, left_on='timestamp', right_on='mov_net_timestamp',how='left')
X = temp_df.drop(columns=['timestamp','mov_net_timestamp'])

kmeans = KMeans(n_clusters=3, random_state=42)

temp_df['cluster_label'] = kmeans.fit_predict(X)
temp_df[['timestamp','cluster_label']]

temp_df.to_csv('/Users/fangsiyu/Desktop/Kolding Hackathon 2026/cluster_kmean_k3.csv', index=False, encoding='utf-8-sig')


In [101]:
temp_df

,timestamp,dur_max_ms_animal,dur_max_ms_bicycle,dur_max_ms_bus,dur_max_ms_car,dur_max_ms_car trailer,dur_max_ms_caravan,dur_max_ms_heavy,dur_max_ms_light,dur_max_ms_motorcycle,...,mov_net_motorcycle,mov_net_pedestrian,mov_net_pram,mov_net_scooter,mov_net_tractor,mov_net_tram,mov_net_truck trailer,mov_net_van,mov_net_wheelchair,cluster_label
0,2026-02-09T01:00:00+01:00,120640.0,21200.0,0.0,208640.0,0.0,0.0,0.0,15720.0,15640.0,...,-1.0,-57.0,2.0,-3.0,0.0,0.0,0.0,-1.0,0.0,1
1,2026-02-10T01:00:00+01:00,46000.0,41600.0,0.0,41840.0,42480.0,0.0,3480.0,49840.0,17720.0,...,0.0,-262.0,13.0,3.0,-1.0,0.0,-1.0,12.0,0.0,2
2,2026-02-11T01:00:00+01:00,33600.0,27480.0,6520.0,32600.0,15160.0,0.0,0.0,32320.0,33440.0,...,-3.0,-387.0,0.0,3.0,0.0,0.0,0.0,-35.0,1.0,2
3,2026-02-12T01:00:00+01:00,26160.0,29320.0,13480.0,27160.0,9360.0,0.0,4120.0,28040.0,21040.0,...,2.0,-309.0,-5.0,-2.0,1.0,0.0,0.0,-6.0,2.0,2
4,2026-02-13T01:00:00+01:00,25240.0,19600.0,22880.0,24040.0,24240.0,0.0,0.0,23240.0,16960.0,...,-3.0,-721.0,2.0,-2.0,0.0,0.0,-1.0,-10.0,1.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,2026-05-04T02:00:00+02:00,8240.0,8640.0,0.0,8640.0,3920.0,0.0,0.0,8600.0,8640.0,...,0.0,-60.0,2.0,0.0,0.0,0.0,0.0,6.0,0.0,0
85,2026-05-05T02:00:00+02:00,8600.0,8240.0,0.0,8600.0,7560.0,0.0,0.0,8600.0,8600.0,...,-2.0,-34.0,-12.0,2.0,0.0,0.0,0.0,9.0,0.0,0
86,2026-05-06T02:00:00+02:00,8320.0,8360.0,0.0,8600.0,7960.0,0.0,0.0,8600.0,8560.0,...,1.0,-42.0,0.0,-2.0,0.0,0.0,0.0,18.0,0.0,0
87,2026-05-07T02:00:00+02:00,8560.0,8360.0,0.0,8560.0,8240.0,0.0,0.0,8480.0,8560.0,...,0.0,57.0,-2.0,0.0,0.0,0.0,0.0,16.0,0.0,0
